# Epileptic Seizure Recognition - Step-by-Step Benchmark Notebook

This notebook is organized as explicit phases so you can run it on:
- **Colab** (cloud)
- **Local machine** (recommended for long runs)

Pipeline stages:
1. Environment setup
2. Run configuration
3. Dataset loading + preprocessing
4. Statistical analysis
5. Cartesian benchmark execution
6. Validation + ranking views
7. 80/20 holdout confusion-matrix comparison
8. Optional export/archive


## Step 1 - Environment Setup (Colab or Local)

`RUN_ENV` is auto-detected by default.
You can still override manually if needed:
- `"local"` for your own laptop/workstation
- `"colab"` for Google Colab


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import platform

# ===== Environment detection =====
IN_COLAB = "google.colab" in sys.modules
RUN_ENV = "colab" if IN_COLAB else "local"  # auto default
# Optional manual override examples:
# RUN_ENV = "local"
# RUN_ENV = "colab"

REPO_URL = "https://github.com/adhamhaithameid/epileptic-seizure-recognition.git"


def find_project_root(start: Path) -> Path:
    """Find project root by walking up until we see the src folder."""
    for cand in [start, *start.parents]:
        if (cand / "src").exists() and (cand / "results").exists():
            return cand
    return start


if RUN_ENV == "colab":
    # Install dependencies in Colab runtime.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "seaborn", "tabulate"],
        check=True,
    )

    repo_dir = Path("/content/epileptic-seizure-recognition")
    if not (repo_dir / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)
    project_dir = repo_dir
else:
    # Local mode: use current notebook location and detect project root automatically.
    project_dir = find_project_root(Path.cwd())

os.chdir(project_dir)
print("RUN_ENV:", RUN_ENV)
print("Project dir:", project_dir)
print("Python:", sys.executable)
print("Platform:", platform.platform())


RUN_ENV: local
Project dir: /Users/adhamhaithameid/Desktop/soft computing - research
Python: /Users/adhamhaithameid/Desktop/soft computing - research/.venv311/bin/python
Platform: macOS-26.4-arm64-arm-64bit


## Step 2 - Run Configuration

Use profiles to control runtime:
- `local_smoke`: fast sanity check
- `local_medium`: practical local run
- `full`: full 4608 fold evaluations

For very long runs, prefer the terminal command:
`python run_all.py --mode cpu --platform-profile linux --non-interactive --fresh`


In [ ]:
import os

# ===== Runtime profile =====
# Auto defaults:
# - local -> local_medium
# - colab -> local_smoke (safe default to avoid long cloud runs)
DEFAULT_PROFILE = "full" if RUN_ENV == "local" else "local_smoke"
RUN_PROFILE = DEFAULT_PROFILE  # local_smoke | local_medium | full

if RUN_PROFILE == "local_smoke":
    MAX_ROWS = 240
    CHECKPOINT_EVERY = 40
    CHECKPOINT_PERCENT = 5
    JOBS = min(4, os.cpu_count() or 1)
    SELECTION_JOBS = min(4, os.cpu_count() or 1)
elif RUN_PROFILE == "local_medium":
    MAX_ROWS = 1200
    CHECKPOINT_EVERY = 80
    CHECKPOINT_PERCENT = 5
    JOBS = max(1, (os.cpu_count() or 1) // 2)
    SELECTION_JOBS = max(1, (os.cpu_count() or 1) // 2)
else:
    MAX_ROWS = None
    CHECKPOINT_EVERY = 120
    CHECKPOINT_PERCENT = 5
    JOBS = max(1, (os.cpu_count() or 1) // 2)
    SELECTION_JOBS = max(1, (os.cpu_count() or 1) // 2)

FRESH_RUN = True
DEVICE_MODE = "cpu"  # cpu | gpu | auto
STRICT_DEVICE = False
ALLOW_PARTIAL_VALIDATION = MAX_ROWS is not None
RUN_LABEL = f"notebook_{RUN_ENV}_{RUN_PROFILE}_{DEVICE_MODE}"

PREPROCESSING_METHODS = ["standard", "minmax", "robust", "quantile"]
REDUCTION_METHODS = ["none", "pca", "lda_projection", "svd"]
SELECTION_METHODS = [
    "none", "filter_chi2", "filter_anova", "filter_correlation",
    "wrapper_sfs", "wrapper_rfe", "embedded_l1", "ga_selection"
]
CLASSIFIER_METHODS = ["knn", "svm", "decision_tree", "logistic_regression", "lda_classifier", "mlp_ann"]
TRACKS = ["binary", "multiclass"]
CV_SPLITS = 3

expected_combos = len(PREPROCESSING_METHODS) * len(REDUCTION_METHODS) * len(SELECTION_METHODS) * len(CLASSIFIER_METHODS) * len(TRACKS)
expected_fold_evals = expected_combos * CV_SPLITS

print("RUN_PROFILE:", RUN_PROFILE)
print("DEVICE_MODE:", DEVICE_MODE)
print("MAX_ROWS:", MAX_ROWS)
print("Expected combos:", expected_combos)
print("Expected fold evals:", expected_fold_evals)


RUN_PROFILE: local_medium
DEVICE_MODE: cpu
MAX_ROWS: 1200
Expected combos: 1536
Expected fold evals: 4608


## Step 3 - Fetch and Load Dataset


In [3]:
import pandas as pd
import numpy as np

from src.config.paths import DATASET_CSV, DATA_PROCESSED_DIR

# Ensure dataset is available (downloads only if needed).
subprocess.run([sys.executable, "src/cli/fetch_data.py"], check=True)

df = pd.read_csv(DATASET_CSV)
unnamed = [c for c in df.columns if c.lower().startswith("unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

target_col = "y" if "y" in df.columns else df.columns[-1]

# Convert features to numeric and keep track of missing values before fill.
for col in df.columns:
    if col != target_col:
        df[col] = pd.to_numeric(df[col], errors="coerce")

X_df = df.drop(columns=[target_col]).copy()
missing_before = int(X_df.isna().sum().sum())
X_df = X_df.fillna(X_df.median(numeric_only=True))
missing_after = int(X_df.isna().sum().sum())

y_multi = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int).to_numpy()
y_binary = (y_multi == 1).astype(int)

DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
X_df.to_csv(DATA_PROCESSED_DIR / "features_numeric.csv", index=False)
pd.DataFrame({"y_multiclass": y_multi, "y_binary": y_binary}).to_csv(DATA_PROCESSED_DIR / "targets.csv", index=False)

print("Dataset loaded:", X_df.shape)
print("Missing before fill:", missing_before)
print("Missing after fill:", missing_after)


Dataset ready.
{
  "rows": 11500,
  "columns": 180,
  "target_column": "y",
  "dropped_bad_rows": 0,
  "path": "data/raw/epileptic_seizure_recognition/epileptic_seizure_data.csv",
  "dataset": "Epileptic Seizure Recognition",
  "source": "Kaggle/UCI source",
  "selected_source_url": "cached_local_file",
  "source_links": [
    "https://raw.githubusercontent.com/akshayg056/Epileptic-seizure-detection-/master/data.csv",
    "https://raw.githubusercontent.com/dragonpilee/Epileptic-Seizure-Detection-System/master/data.csv",
    "https://archive.ics.uci.edu/static/public/388/epileptic+seizure+recognition.zip"
  ]
}
Dataset loaded: (11500, 178)
Missing before fill: 0
Missing after fill: 0


## Step 4 - Statistical Analysis (Descriptive Stats, Covariance, Correlation, Heatmap)


In [4]:
from scipy.stats import kurtosis, skew
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.paths import RESULTS_TABLES_DIR, RESULTS_FIGURES_DIR

RESULTS_TABLES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

desc = pd.DataFrame({
    "min": X_df.min(),
    "max": X_df.max(),
    "mean": X_df.mean(),
    "variance": X_df.var(),
    "std": X_df.std(),
    "skewness": X_df.apply(lambda s: skew(s.to_numpy(), bias=False)),
    "kurtosis": X_df.apply(lambda s: kurtosis(s.to_numpy(), bias=False)),
})
desc.to_csv(RESULTS_TABLES_DIR / "dataset_descriptive_stats.csv")

cov = X_df.cov()
corr = X_df.corr()
cov.to_csv(RESULTS_TABLES_DIR / "covariance_matrix.csv")
corr.to_csv(RESULTS_TABLES_DIR / "correlation_matrix.csv")

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(RESULTS_FIGURES_DIR / "correlation_heatmap.png", dpi=220)
plt.close()

print("Statistical outputs saved to results/tables and results/figures")


Statistical outputs saved to results/tables and results/figures


## Step 5 - Run Cartesian Benchmark


In [5]:
from src.runtime.acceleration import configure_acceleration
from src.core import CartesianSpec, RunnerIO, run_cartesian_benchmark, build_summary, save_comparisons, generate_cartesian_plots
from src.config.paths import RESULTS_METRICS_DIR, RESULTS_TABLES_DIR, RESULTS_REPORTS_DIR, RESULTS_FIGURES_DIR

accel = configure_acceleration(device=DEVICE_MODE, strict=STRICT_DEVICE)
print(f"Acceleration: requested={accel.requested} resolved={accel.resolved} backend={accel.backend} gpu={accel.gpu_count}")
print(accel.message)

spec = CartesianSpec(
    preprocessing=tuple(PREPROCESSING_METHODS),
    reduction=tuple(REDUCTION_METHODS),
    selection=tuple(SELECTION_METHODS),
    classifiers=tuple(CLASSIFIER_METHODS),
    tracks=tuple(TRACKS),
    cv_splits=CV_SPLITS,
)

metrics_csv = RESULTS_METRICS_DIR / "cartesian_metrics_all.csv"
manifest_json = RESULTS_METRICS_DIR / "cartesian_run_manifest.json"
if FRESH_RUN:
    metrics_csv.unlink(missing_ok=True)
    manifest_json.unlink(missing_ok=True)

io = RunnerIO(metrics_csv=metrics_csv, manifest_json=manifest_json, checkpoint_every=CHECKPOINT_EVERY)
metrics_df = run_cartesian_benchmark(
    X_df=X_df,
    y_binary=y_binary,
    y_multiclass=y_multi,
    spec=spec,
    io=io,
    resume=not FRESH_RUN,
    max_rows=MAX_ROWS,
    model_jobs=max(1, JOBS),
    selection_jobs=max(1, SELECTION_JOBS),
    execution_device=accel.resolved,
    acceleration_backend=accel.backend,
    checkpoint_percent=CHECKPOINT_PERCENT,
    run_label=RUN_LABEL,
    platform_profile=RUN_ENV,
)

ok = metrics_df[metrics_df["status"] == "ok"].copy()
summary = build_summary(ok).sort_values(["track", "accuracy"], ascending=[True, False])
summary.to_csv(RESULTS_TABLES_DIR / "cartesian_summary_by_combo.csv", index=False)

saved = save_comparisons(summary, RESULTS_TABLES_DIR, RESULTS_REPORTS_DIR)
plots = generate_cartesian_plots(metrics_df, summary, RESULTS_FIGURES_DIR, X_df, y_binary, CV_SPLITS)

print("Run completed")
print("Metrics rows:", len(metrics_df))
print("Saved plots:", len(plots))


Acceleration: requested=cpu resolved=cpu backend=none gpu=0
CPU mode selected.


/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[checkpoint] 5% complete (60/1200) elapsed=00:01:07 eta=00:21:26


/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(


[checkpoint] 10% complete (120/1200) elapsed=00:01:42 eta=00:15:26


...

[checkpoint] 90% complete (1080/1200) elapsed=00:15:07 eta=00:01:40


/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(


[checkpoint] 95% complete (1140/1200) elapsed=00:15:41 eta=00:00:49


/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (350) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/adhamhaithameid/Desktop/soft computing - research/.venv311/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
 

[checkpoint] 100% complete (1200/1200) elapsed=00:16:47 eta=00:00:00
Run completed
Metrics rows: 1200
Saved plots: 6


## Step 6 - Validate Outputs and Inspect Rankings


In [6]:
import json
from IPython.display import display

validate_cmd = [sys.executable, "src/cli/validate_cartesian_outputs.py"]
if ALLOW_PARTIAL_VALIDATION:
    validate_cmd.append("--allow-partial")
subprocess.run(validate_cmd, check=True)

manifest = json.loads((Path("results/metrics/cartesian_run_manifest.json")).read_text(encoding="utf-8"))
print(json.dumps(manifest, indent=2))

display(summary[summary["track"] == "binary"].head(10))
display(summary[summary["track"] == "multiclass"].head(10))


Python(1328) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Validation passed.
Report written: /Users/adhamhaithameid/Desktop/soft computing - research/results/reports/cartesian_validation_report.md
{
  "expected_combos": 1536,
  "expected_fold_evals": 4608,
  "target_total_rows": 1200,
  "rows_written": 1200,
  "completed_ok": 1128,
  "skipped_or_failed": 72,
  "runtime_sec": 1007.0996940420009,
  "runtime_sec_last_invocation": 1007.0996940420009,
  "started_utc": "2026-04-10T15:08:44.498488+00:00",
  "finished_utc": "2026-04-10T15:25:31.616417+00:00",
  "checkpoint_percent": 5,
  "resume_mode": false,
  "resume_noop": false,
  "max_rows": 1200,
  "run_label": "notebook_local_local_medium_cpu",
  "platform_profile": "local",
  "execution_device": "cpu",
  "acceleration_backend": "none",
  "best_binary": {
    "track": "binary",
    "preprocessing": "quantile",
    "reduction": "pca",
    "selection": "none",
    "model": "svm",
    "accuracy": 0.9796557120500784,
    "precision": 0.9636608344549124,
    "recall": 0.9335071707953064,
    "f1": 

,track,preprocessing,reduction,selection,model,accuracy,precision,recall,f1,roc_auc,fit_time_sec,predict_time_sec
299,binary,quantile,pca,none,svm,0.979656,0.963661,0.933507,0.948344,0.996806,1.336633,0.170581
251,binary,quantile,none,none,svm,0.979395,0.964865,0.930900,0.947578,0.997162,11.317389,0.391066
347,binary,quantile,svd,none,svm,0.977569,0.960758,0.925684,0.942895,0.996192,1.681869,0.229507
221,binary,quantile,none,embedded_l1,svm,0.976787,0.956873,0.925684,0.941021,0.996756,2.642386,0.293242
479,binary,robust,pca,none,svm,0.973918,0.956224,0.911343,0.933244,0.996413,2.128121,0.189300
166,binary,minmax,svd,none,mlp_ann,0.973784,0.924750,0.945855,0.935176,0.510611,3.680557,0.001523
527,binary,robust,svd,none,svm,0.973396,0.954856,0.910039,0.931909,0.996247,1.485010,0.172682
245,binary,quantile,none,ga_selection,svm,0.972613,0.952186,0.908735,0.929953,0.995226,1.228385,0.189382
526,binary,robust,svd,none,mlp_ann,0.972353,0.944818,0.915254,0.929801,0.987047,1.552627,0.001154
118,binary,minmax,pca,none,mlp_ann,0.971957,0.932385,0.926934,0.929639,0.991803,2.500955,0.001009


,track,preprocessing,reduction,selection,model,accuracy,precision,recall,f1,roc_auc,fit_time_sec,predict_time_sec


## Step 7 - 80/20 Holdout Classifier + Confusion Matrix Table


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from src.core.cartesian_pipeline import build_classifier, CLASSIFIER_METHODS

X_train, X_test, yb_train, yb_test = train_test_split(
    X_df, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

rows = []
for model_name in CLASSIFIER_METHODS:
    # Standard scaling baseline for fair 80/20 comparison.
    model = make_pipeline(StandardScaler(), build_classifier(model_name))
    model.fit(X_train, yb_train)
    pred = model.predict(X_test)

    auc = np.nan
    if hasattr(model, "predict_proba"):
        auc = roc_auc_score(yb_test, model.predict_proba(X_test)[:, 1])
    elif hasattr(model, "decision_function"):
        auc = roc_auc_score(yb_test, model.decision_function(X_test))

    tn, fp, fn, tp = confusion_matrix(yb_test, pred, labels=[0, 1]).ravel()
    rows.append({
        "model": model_name,
        "accuracy": accuracy_score(yb_test, pred),
        "error_rate": 1 - accuracy_score(yb_test, pred),
        "precision": precision_score(yb_test, pred, zero_division=0),
        "recall": recall_score(yb_test, pred, zero_division=0),
        "f1": f1_score(yb_test, pred, zero_division=0),
        "roc_auc": auc,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    })

holdout_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
holdout_df.to_csv("results/tables/holdout_80_20_binary_classifier_comparison.csv", index=False)
display(holdout_df)


,model,accuracy,error_rate,precision,recall,f1,roc_auc,tn,fp,fn,tp
5,mlp_ann,0.974348,0.025652,0.954649,0.915217,0.934517,0.988651,1820,20,39,421
1,svm,0.970870,0.029130,0.953811,0.897826,0.924972,0.996471,1820,20,47,413
2,decision_tree,0.942174,0.057826,0.897810,0.802174,0.847302,0.859244,1798,42,91,369
0,knn,0.931739,0.068261,0.993485,0.663043,0.795306,0.921806,1838,2,155,305
4,lda_classifier,0.821304,0.178696,0.929825,0.115217,0.205029,0.497884,1836,4,407,53
3,logistic_regression,0.815652,0.184348,0.950000,0.082609,0.152000,0.499745,1838,2,422,38


## Step 8 - Optional Export (Colab)


In [8]:
if RUN_ENV == "colab":
    subprocess.run(["zip", "-rq", "colab_outputs.zip", "results", "paper/draft", "data/processed"], check=True)
    print("Created colab_outputs.zip")
else:
    print("Local mode: no zip export needed.")


Local mode: no zip export needed.
